In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

In [2]:
class Bottleneck(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        mid = out_channels // 4

        self.conv1 = nn.Conv2d(
            in_channels, mid, 1
        )

        # Depthwise convolution
        self.conv2 = nn.Conv2d(
            mid, mid, 3,
            padding=1,
            groups=mid
        )

        self.conv3 = nn.Conv2d(
            mid, out_channels, 1
        )

        self.skip = nn.Conv2d(
            in_channels,
            out_channels,
            1
        )

        self.relu = nn.ReLU()

    def forward(self, x):

        identity = self.skip(x)

        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.conv3(x)

        return self.relu(x + identity)

In [3]:
x = torch.randn(2, 64, 32, 32)

block = Bottleneck(64, 256)

output = block(x)

print("Input shape :", x.shape)
print("Output shape:", output.shape)

Input shape : torch.Size([2, 64, 32, 32])
Output shape: torch.Size([2, 256, 32, 32])


In [4]:
params = sum(
    p.numel()
    for p in block.parameters()
)

print("Total parameters:", params)

Total parameters: 38080


In [5]:
from torch.profiler import profile, ProfilerActivity

with profile(
    activities=[ProfilerActivity.CPU],
    record_shapes=True
) as prof:

    block(x)

print(
    prof.key_averages().table(
        sort_by="cpu_time_total",
        row_limit=10
    )
)

------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                          Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                  aten::conv2d         0.27%      37.239us        65.57%       9.007ms       2.252ms             4  
             aten::convolution         0.81%     111.949us        65.30%       8.970ms       2.243ms             4  
            aten::_convolution         0.49%      67.643us        64.49%       8.858ms       2.215ms             4  
             aten::thnn_conv2d         0.13%      17.451us        59.88%       8.226ms       2.742ms             3  
    aten::_slow_conv2d_forward        49.07%       6.740ms        59.76%       8.208ms       2.736ms             3  
                    aten::relu        13.31%       1.829ms      

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


In [6]:
for channels in [128, 256, 512]:

    block = Bottleneck(64, channels)

    params = sum(
        p.numel()
        for p in block.parameters()
    )

    print(
        f"Channels: {channels}, "
        f"Parameters: {params}"
    )

Channels: 128, Parameters: 14944
Channels: 256, Parameters: 38080
Channels: 512, Parameters: 108928


In [7]:
print("ResNet-style bottleneck completed.")
print("Depthwise convolution, skip projection and profiling were demonstrated.")

ResNet-style bottleneck completed.
Depthwise convolution, skip projection and profiling were demonstrated.
